In [ ]:
# pip install some library
%pip install einops matplotlib jupyter_black scikit_learn

In [ ]:
%load_ext jupyter_black
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from einops import rearrange, reduce

In [ ]:
from data_loader import SITS, SITSDataSet, collate_fn, narrow_tensor, split_sits
from transformer import Transformer

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import s3fs

# Code to connect to share data in S3
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": S3_ENDPOINT_URL})
# now you should be able to see my share repository "mfauvel/diffusion/UNET_DATA
# Wae are going to copy locally the data set
data_dir = Path.home() / "SITS"
data_dir.mkdir(exist_ok=True, parents=True)
fs.get(f"s3://mfauvel/diffusion/SITS/*", data_dir)

In [ ]:
# Define Parameters
BATCH_SIZE = 8
NUM_WORKERS = 8
N_CHANNELS = 10
DEPTH = 1
EMB_DIM = 64
DEVICE = torch.device("cuda")
N_EPOCHS = 20
LEARNING_RATE = 0.005
DTYPE = torch.float32

In [ ]:
# Init data set
data_dir = Path.home() / "SITS"
data = SITSDataSet(data_dir, dtype=DTYPE, init_scaler=True)
train_data, valid_data, test_data = random_split(data, [0.6, 0.2, 0.2])

# Init data loader
train_data_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
)
valid_data_loader = DataLoader(
    valid_data,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
)
test_data_loader = DataLoader(
    test_data, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, collate_fn=collate_fn
)

In [ ]:
# Init model
model = Transformer(
    n_channels=N_CHANNELS, emb_dim=EMB_DIM, device=DEVICE, dtype=DTYPE, depth=DEPTH
)
model.compile()

In [ ]:
# Init optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.MSELoss()
train_loss, valid_loss = [], []
best_loss = torch.inf

In [ ]:
# Train loop
for epoch in range(N_EPOCHS):
    model.train()
    accu = 0.0
    for sits in train_data_loader:
        sits = sits.to(DEVICE)

        # Split data w.r.t. time
        sits_unmasked, _ = split_sits(sits)

        # Convert to nested tensors
        sits_unmasked = SITS(
            data=torch.nested.nested_tensor(
                narrow_tensor(sits_unmasked.data, sits_unmasked.mask),
                layout=torch.jagged,
            ),
            doy=torch.nested.nested_tensor(
                narrow_tensor(sits_unmasked.doy, sits_unmasked.mask),
                layout=torch.jagged,
            ),
        )
        sits = SITS(
            data=torch.nested.nested_tensor(
                narrow_tensor(sits.data, sits.mask),
                layout=torch.jagged,
            ),
            doy=torch.nested.nested_tensor(
                narrow_tensor(sits.doy, sits.mask),
                layout=torch.jagged,
            ),
        )

        # forward pass
        pred = model(sits_unmasked.data, sits_unmasked.doy, sits.doy)
        loss = loss_fn(
            pred.values(), sits.data.values()
        )  # jagged dim are equal but are not the same !

        # Backward
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # log loss
        accu += loss.item()
    train_loss.append(accu / len(train_data_loader))
    print(f"Epoch: {epoch} \ntrain loss={train_loss[-1]}")

    model.eval()
    accu = 0.0
    with torch.no_grad():
        for sits in valid_data_loader:
            sits = sits.to(DEVICE)

            # Split data w.r.t. time
            sits_unmasked, sits_masked = split_sits(sits)

            # Convert to nested tensors
            sits_unmasked = SITS(
                data=torch.nested.nested_tensor(
                    narrow_tensor(sits_unmasked.data, sits_unmasked.mask),
                    layout=torch.jagged,
                ),
                doy=torch.nested.nested_tensor(
                    narrow_tensor(sits_unmasked.doy, sits_unmasked.mask),
                    layout=torch.jagged,
                ),
            )
            sits = SITS(
                data=torch.nested.nested_tensor(
                    narrow_tensor(sits.data, sits.mask),
                    layout=torch.jagged,
                ),
                doy=torch.nested.nested_tensor(
                    narrow_tensor(sits.doy, sits.mask),
                    layout=torch.jagged,
                ),
            )

            # forward pass
            pred = model(sits_unmasked.data, sits_unmasked.doy, sits.doy)
            loss = loss_fn(
                pred.values(), sits.data.values()
            )  # jagged dim are equal but are not the same !

            # log loss
            accu += loss.item()

        valid_loss.append(accu / len(valid_data_loader))
        if best_loss > valid_loss[-1]:
            best_loss = valid_loss[-1]
            torch.save(model.state_dict(), "best_model.pt")
        print(f"valid loss={valid_loss[-1]}")



In [ ]:
plt.plot(train_loss)
plt.plot(valid_loss)